In [77]:
!pip install requests python-dotenv

In [78]:
import pandas as pd
import requests
import os
from dotenv import load_dotenv

In [79]:
load_dotenv()
API_KEY = os.getenv("OPENWEATHER_API_KEY")


In [80]:
cities = ["Lucknow", "Delhi", "Mumbai", "Chennai", "Shimla"]
all_data = []

In [81]:
for city in cities:
    url = f"https://api.openweathermap.org/data/2.5/weather?q={city},IN&appid={API_KEY}&units=metric"
    data = requests.get(url).json()
    all_data.append(data)

In [82]:
df = pd.json_normalize(all_data).head()
df["weather"]


0    [{'id': 800, 'main': 'Clear', 'description': '...
1    [{'id': 800, 'main': 'Clear', 'description': '...
2    [{'id': 800, 'main': 'Clear', 'description': '...
3    [{'id': 801, 'main': 'Clouds', 'description': ...
4    [{'id': 802, 'main': 'Clouds', 'description': ...
Name: weather, dtype: object

In [83]:

df["weather_main"] = df["weather"].str[0].str["main"]

In [84]:
cols = ['dt','id', 'name', 'cod',
        'coord.lon', 'coord.lat', 'main.temp',
        'main.temp_min', 'main.temp_max', 'main.pressure',
       'main.humidity', 'main.sea_level', 'main.grnd_level', 'wind.speed','rain.1h','weather_main'
       ]
df = df.reindex(columns = cols)
df

,dt,id,name,cod,coord.lon,coord.lat,main.temp,main.temp_min,main.temp_max,main.pressure,main.humidity,main.sea_level,main.grnd_level,wind.speed,rain.1h,weather_main
0,1791523137,1264733,Lucknow,200,80.9167,26.8500,25.37,25.37,25.37,1014,81,1014,1000,2.50,NaN,Clear
1,1791522684,1273294,Delhi,200,77.2167,28.6667,28.51,28.51,28.51,1014,57,1014,988,2.13,NaN,Clear
2,1791522762,1275339,Mumbai,200,72.8479,19.0144,29.78,29.78,29.78,1014,64,1014,1014,3.19,NaN,Clear
3,1791522778,1264527,Chennai,200,80.2785,13.0878,33.42,33.42,33.42,1014,68,1014,1013,3.72,NaN,Clouds
4,1791523278,1256237,Shimla,200,77.1667,31.1000,16.69,16.69,16.69,1016,71,1016,854,1.39,NaN,Clouds


In [85]:
df = df.rename(columns = {"name": "city",
    "main.temp": "temp_c",
    "main.humidity": "humidity",
    "main.pressure": "pressure",
    "wind.speed": "wind_speed",
    "rain.1h": "rain_1h",
    "coord.lat": "lat",
    "coord.lon": "lon",
                    })
df

,dt,id,city,cod,lon,lat,temp_c,main.temp_min,main.temp_max,pressure,humidity,main.sea_level,main.grnd_level,wind_speed,rain_1h,weather_main
0,1791523137,1264733,Lucknow,200,80.9167,26.8500,25.37,25.37,25.37,1014,81,1014,1000,2.50,NaN,Clear
1,1791522684,1273294,Delhi,200,77.2167,28.6667,28.51,28.51,28.51,1014,57,1014,988,2.13,NaN,Clear
2,1791522762,1275339,Mumbai,200,72.8479,19.0144,29.78,29.78,29.78,1014,64,1014,1014,3.19,NaN,Clear
3,1791522778,1264527,Chennai,200,80.2785,13.0878,33.42,33.42,33.42,1014,68,1014,1013,3.72,NaN,Clouds
4,1791523278,1256237,Shimla,200,77.1667,31.1000,16.69,16.69,16.69,1016,71,1016,854,1.39,NaN,Clouds


In [86]:
df.isnull().sum()

dt                 0
id                 0
city               0
cod                0
lon                0
lat                0
temp_c             0
main.temp_min      0
main.temp_max      0
pressure           0
humidity           0
main.sea_level     0
main.grnd_level    0
wind_speed         0
rain_1h            5
weather_main       0
dtype: int64

In [87]:
df["rain_1h"] = df["rain_1h"].fillna(0)
df["timestamp_utc"] = pd.to_datetime(df["dt"], unit="s")

In [88]:
df

,dt,id,city,cod,lon,lat,temp_c,main.temp_min,main.temp_max,pressure,humidity,main.sea_level,main.grnd_level,wind_speed,rain_1h,weather_main,timestamp_utc
0,1791523137,1264733,Lucknow,200,80.9167,26.8500,25.37,25.37,25.37,1014,81,1014,1000,2.50,0.0,Clear,2026-10-09 05:18:57
1,1791522684,1273294,Delhi,200,77.2167,28.6667,28.51,28.51,28.51,1014,57,1014,988,2.13,0.0,Clear,2026-10-09 05:11:24
2,1791522762,1275339,Mumbai,200,72.8479,19.0144,29.78,29.78,29.78,1014,64,1014,1014,3.19,0.0,Clear,2026-10-09 05:12:42
3,1791522778,1264527,Chennai,200,80.2785,13.0878,33.42,33.42,33.42,1014,68,1014,1013,3.72,0.0,Clouds,2026-10-09 05:12:58
4,1791523278,1256237,Shimla,200,77.1667,31.1000,16.69,16.69,16.69,1016,71,1016,854,1.39,0.0,Clouds,2026-10-09 05:21:18


In [90]:
df = df[["city", "timestamp_utc", "temp_c", "humidity", "pressure",
         "wind_speed", "rain_1h", "weather_main", "lat", "lon"]]
df

,city,timestamp_utc,temp_c,humidity,pressure,wind_speed,rain_1h,weather_main,lat,lon
0,Lucknow,2026-10-09 05:18:57,25.37,81,1014,2.50,0.0,Clear,26.8500,80.9167
1,Delhi,2026-10-09 05:11:24,28.51,57,1014,2.13,0.0,Clear,28.6667,77.2167
2,Mumbai,2026-10-09 05:12:42,29.78,64,1014,3.19,0.0,Clear,19.0144,72.8479
3,Chennai,2026-10-09 05:12:58,33.42,68,1014,3.72,0.0,Clouds,13.0878,80.2785
4,Shimla,2026-10-09 05:21:18,16.69,71,1016,1.39,0.0,Clouds,31.1000,77.1667


In [91]:
import sqlite3

conn = sqlite3.connect("climate.db")
df.to_sql("weather", conn, if_exists="append", index=False)

pd.read_sql("SELECT * FROM weather", conn)

,city,timestamp_utc,temp_c,humidity,pressure,wind_speed,rain_1h,weather_main,lat,lon
0,Lucknow,2026-10-09 05:18:57,25.37,81,1014,2.50,0.0,Clear,26.8500,80.9167
1,Delhi,2026-10-09 05:11:24,28.51,57,1014,2.13,0.0,Clear,28.6667,77.2167
2,Mumbai,2026-10-09 05:12:42,29.78,64,1014,3.19,0.0,Clear,19.0144,72.8479
3,Chennai,2026-10-09 05:12:58,33.42,68,1014,3.72,0.0,Clouds,13.0878,80.2785
4,Shimla,2026-10-09 05:21:18,16.69,71,1016,1.39,0.0,Clouds,31.1000,77.1667
